# Exercise 3: logistic regression on real human promoter sequences

Use **human_nontata_promoters** from **Genomic Benchmarks** to classify real human DNA sequences.
Label **1 = non-TATA promoter**; label **0 = non-promoter**.
Each sequence is 251 nt long. These are real genomic sequences, not the simulated sequences from Exercise 1.

We will keep the benchmark's **official test set of 9,034 sequences** unchanged.
From its **27,097 training sequences**, randomly reserve 20% of each label for validation.
This gives 21,678 training, 5,419 validation, and 9,034 test sequences (approximately 60% / 15% / 25% overall).

As in Exercise 2, walk through **k = 4**, then compare **k = 1 through 10**.
Choose k using validation ROC-AUC, and evaluate only the selected model on the test set.
Fit the vocabulary and logistic regression on training sequences only.

Sources:
- [Genomic Benchmarks: dataset description and provenance](https://github.com/ML-Bioinfo-CEITEC/genomic_benchmarks/tree/main/docs/human_nontata_promoters)
- [Maintainer-hosted Hugging Face dataset](https://huggingface.co/datasets/katarinagresova/Genomic_Benchmarks_human_nontata_promoters)

The notebook downloads public data directly. No CSV upload or other project files are required.


## Install the Parquet reader

Run this cell once. The exercise also uses NumPy, pandas, scikit-learn, and Matplotlib.
If those libraries are missing, install them with `%pip install numpy pandas scikit-learn matplotlib`.


In [ ]:
%pip install -q pyarrow


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

seed = 42
threshold = 0.5


## 1. Download the official training and test sets

Read the maintainer-hosted Parquet files with pandas. The URL names a fixed dataset revision
so later changes to the repository do not silently change this exercise's data.
Rename the `seq` column to `sequence`, matching Exercise 2.

Do not combine or reshuffle the official test set with the training set.
The classes are slightly imbalanced; we keep the supplied class proportions.

`N` means an unknown nucleotide. One official test sequence contains `N`.
We retain it, without replacing or removing it. A k-mer containing `N` will be ignored
if it does not appear in the training-fitted vocabulary, just like any unseen k-mer.


In [ ]:
data_url = (
    "https://huggingface.co/datasets/katarinagresova/"
    "Genomic_Benchmarks_human_nontata_promoters/resolve/"
    "e0003669df0a180c8570ebe091ed91f20fa080ec/data/"
)
benchmark_train = pd.read_parquet(data_url + "train-00000-of-00001-aeb55728756a44ae.parquet")
test_data = pd.read_parquet(data_url + "test-00000-of-00001-d7614d9116c62f44.parquet")

benchmark_train = benchmark_train.rename(columns={"seq": "sequence"})
test_data = test_data.rename(columns={"seq": "sequence"})

assert benchmark_train["sequence"].str.fullmatch("[ACGTN]{251}").all()
assert test_data["sequence"].str.fullmatch("[ACGTN]{251}").all()
assert set(benchmark_train["label"]) == {0, 1}
assert set(test_data["label"]) == {0, 1}

print("Official training sequences:", len(benchmark_train))
print("Official test sequences (reserved):", len(test_data))
print("Official training class counts:")
print(benchmark_train["label"].value_counts())
benchmark_train.head()


## 2. Reserve validation sequences from the official training set

For each label, randomly select **20% of the official training sequences for validation**.
The remaining 80% are used for training. The supplied test set is not changed.

`groupby("label").sample(frac=0.20, ...)` samples the same fraction from each label.
`drop(...)` removes those rows from training, so training and validation cannot overlap.

This is **not** the 70/20/10 split from Exercise 2. We preserve the benchmark's official test set.
Use this same split for every k. Do not fit the vocabulary or model on validation or test sequences.


In [ ]:
validation_data = benchmark_train.groupby("label").sample(frac=0.20, random_state=seed)
train_data = benchmark_train.drop(validation_data.index)

y_train = train_data["label"]
y_validation = validation_data["label"]

print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Official test (reserved):", len(test_data))
print("Validation class counts:")
print(validation_data["label"].value_counts())


## 3. Moving window -> bag of k-mers

A k-mer is a DNA substring of length k. Move a window one base at a time,
including overlapping windows. For `ACACAC` and k = 2, the windows are
`AC`, `CA`, `AC`, `CA`, `AC`. The bag contains counts: AC = 3, CA = 2.

We keep **counts**, but discard where each k-mer occurred and the order of the windows.
For a 251-nt sequence, there are 251 - k + 1 windows.

`CountVectorizer(analyzer="char", ngram_range=(k, k))` performs this moving-window counting.
It stores a sparse matrix: rows are sequences, columns are training-observed k-mers,
and entries are counts. An unseen k-mer is ignored when transforming validation or test sequences.
We do not merge reverse complements or use TF-IDF.

In [ ]:
example = "ACACAC"
example_k = 2
windows = []

for start in range(len(example) - example_k + 1):
    windows.append(example[start:start + example_k])

print("Moving windows:", windows)
pd.Series(windows).value_counts()


## 4. Walk through one model: k = 4

Before comparing all k values, fit and inspect a single example.
Change `example_k` below if you want to walk through a different k.

1. Count overlapping k-mers, fitting the vocabulary on training sequences only.
2. Fit logistic regression on training counts and labels.
3. Predict validation probabilities and apply the fixed 0.5 threshold.
4. Read the metrics, ROC curve, precision-recall curve, and confusion matrix.

This cell does not use the test set. The next loop repeats the same workflow for k = 1 through 10.


In [ ]:
example_k = 4

example_vectorizer = CountVectorizer(
    analyzer="char", ngram_range=(example_k, example_k), lowercase=False, dtype=np.int32,
)
x_example_train = example_vectorizer.fit_transform(train_data["sequence"])
x_example_validation = example_vectorizer.transform(validation_data["sequence"])
print("Training count matrix:", x_example_train.shape)
print("Validation count matrix:", x_example_validation.shape)

example_model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000, random_state=seed)
example_model.fit(x_example_train, y_train)
example_probabilities = example_model.predict_proba(x_example_validation)[:, 1]
example_predictions = (example_probabilities >= threshold).astype(int)

example_results = pd.DataFrame([{
    "k": example_k,
    "features": x_example_train.shape[1],
    "accuracy": accuracy_score(y_validation, example_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_validation, example_predictions),
    "precision": precision_score(y_validation, example_predictions, zero_division=0),
    "recall": recall_score(y_validation, example_predictions, zero_division=0),
    "f1": f1_score(y_validation, example_predictions, zero_division=0),
    "roc_auc": roc_auc_score(y_validation, example_probabilities),
    "average_precision": average_precision_score(y_validation, example_probabilities),
    "log_loss": log_loss(y_validation, example_probabilities),
}]).set_index("k")
print("Validation metrics:")
print(example_results.round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
RocCurveDisplay.from_predictions(
    y_validation, example_probabilities, ax=axes[0], name=f"k={example_k}",
)
axes[0].plot([0, 1], [0, 1], "k--")
PrecisionRecallDisplay.from_predictions(
    y_validation, example_probabilities, ax=axes[1], name=f"k={example_k}",
)
axes[1].axhline(y_validation.mean(), color="black", linestyle="--")
ConfusionMatrixDisplay.from_predictions(
    y_validation, example_predictions, labels=[0, 1], display_labels=["Negative", "Positive"],
    ax=axes[2], colorbar=False, cmap="Blues",
)
axes[0].set_title(f"Validation ROC, k={example_k}")
axes[1].set_title(f"Validation precision-recall, k={example_k}")
axes[2].set_title(f"Validation confusion matrix, k={example_k}")
plt.tight_layout()
plt.show()


## 5. Fit one logistic regression for each k

Fit the vocabulary on training sequences with `fit_transform`.
Use only `transform` on validation sequences, so they cannot add vocabulary entries.

Keep the model settings fixed: L2 regularization with C = 1 and the `liblinear` solver.
Only k changes. Larger k can create many more features and take longer to fit.
We keep the sparse matrix rather than turning it into a large dense table.

Store validation probabilities for every k. Keep the training-fitted model and vocabulary
with the highest **validation ROC-AUC**. If scores tie exactly, keep the smaller k.
This fixes the selection rule before looking at the test set.

In [ ]:
validation_probabilities = {}
feature_counts = {}
best_validation_auc = -1

for k in range(1, 11):
    vectorizer = CountVectorizer(
        analyzer="char", ngram_range=(k, k), lowercase=False, dtype=np.int32,
    )
    x_train = vectorizer.fit_transform(train_data["sequence"])
    x_validation = vectorizer.transform(validation_data["sequence"])

    model = LogisticRegression(C=1.0, solver="liblinear", max_iter=1000, random_state=seed)
    model.fit(x_train, y_train)

    probabilities = model.predict_proba(x_validation)[:, 1]
    validation_probabilities[k] = probabilities
    feature_counts[k] = x_train.shape[1]
    validation_auc = roc_auc_score(y_validation, probabilities)

    if validation_auc > best_validation_auc:
        best_validation_auc = validation_auc
        best_k = k
        best_model = model
        best_vectorizer = vectorizer

    print(f"k={k:2d} | features={feature_counts[k]:7,d} | validation ROC-AUC={validation_auc:.4f}")


## 6. Compare validation metrics for every k

- **Accuracy:** fraction of correct class predictions.
- **Balanced accuracy:** mean recall across the two classes.
- **Precision:** among predicted positives, the fraction that are positive.
- **Recall:** among actual positives, the fraction predicted positive.
- **F1:** harmonic mean of precision and recall.
- **ROC-AUC:** ranking quality across thresholds; higher is better, and 0.5 is chance-level ranking.
- **Average precision (AP):** a summary of the precision-recall curve; higher is better. This is not trapezoidal PR-AUC.
- **Log loss:** error in the predicted probabilities; lower is better.

Accuracy, balanced accuracy, precision, recall, F1, and confusion matrices use the fixed
threshold of 0.5. ROC-AUC and AP use probabilities across thresholds, not the class calls.
If a model makes no positive calls, report precision as 0 (`zero_division=0`).

In [ ]:
rows = []

for k in range(1, 11):
    probabilities = validation_probabilities[k]
    predictions = (probabilities >= threshold).astype(int)
    rows.append({
        "k": k,
        "features": feature_counts[k],
        "accuracy": accuracy_score(y_validation, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_validation, predictions),
        "precision": precision_score(y_validation, predictions, zero_division=0),
        "recall": recall_score(y_validation, predictions, zero_division=0),
        "f1": f1_score(y_validation, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_validation, probabilities),
        "average_precision": average_precision_score(y_validation, probabilities),
        "log_loss": log_loss(y_validation, probabilities),
    })

validation_results = pd.DataFrame(rows).set_index("k")
validation_results.round(4)


## 7. Compare ROC and precision-recall curves

Each curve uses the same validation sequences. The ROC diagonal is chance-level ranking.
The precision-recall reference line is the positive-class fraction in the validation set.
The legends show ROC-AUC and average precision for each k.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for k in range(1, 11):
    RocCurveDisplay.from_predictions(
        y_validation, validation_probabilities[k], name=f"k={k}", ax=axes[0],
    )
    PrecisionRecallDisplay.from_predictions(
        y_validation, validation_probabilities[k], name=f"k={k}", ax=axes[1],
    )

axes[0].plot([0, 1], [0, 1], "k--", label="Chance")
axes[1].axhline(y_validation.mean(), color="black", linestyle="--", label="Class prevalence")
axes[0].set_title("Validation ROC curves")
axes[1].set_title("Validation precision-recall curves")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## 8. Compare validation confusion matrices

Rows are actual labels; columns are predicted labels. Each matrix shows true negatives,
false positives, false negatives, and true positives at the same 0.5 threshold.
The shared color scale makes the matrices comparable.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for k, ax in zip(range(1, 11), axes.flat):
    predictions = (validation_probabilities[k] >= threshold).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_validation, predictions, labels=[0, 1], display_labels=["Negative", "Positive"],
        ax=ax, colorbar=False, cmap="Blues", im_kw={"vmin": 0, "vmax": y_validation.value_counts().max()},
    )
    ax.set_title(f"Validation, k={k}")

plt.tight_layout()
plt.show()


## 9. Select k using validation results only

The selection rule is the largest validation ROC-AUC, not whichever metric looks best afterwards.
Discuss the feature counts, scores, and curves before continuing.
The selected model is still fitted on the 70% training split only; we do not refit it on validation data.

In [ ]:
print("Selected k:", best_k)
print("Validation ROC-AUC:", round(best_validation_auc, 4))
validation_results.loc[[best_k]].round(4)


## 10. STOP before the official test evaluation

Finish choosing k and discussing validation results **before running the cell below**.
**Run all also runs this final test evaluation.** Run cells individually if teaching the split boundaries.

Evaluate only the selected model on the unchanged official test set of 9,034 sequences.
Transform with its training-fitted vocabulary; do not fit or choose anything using the test results.
The selected model remains fitted on the training subset only, without a validation-data refit.
Do not change k based on the test results.


In [ ]:
y_test = test_data["label"]
x_test = best_vectorizer.transform(test_data["sequence"])
test_probabilities = best_model.predict_proba(x_test)[:, 1]
test_predictions = (test_probabilities >= threshold).astype(int)

test_results = pd.DataFrame([{
    "k": best_k,
    "accuracy": accuracy_score(y_test, test_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_predictions),
    "precision": precision_score(y_test, test_predictions, zero_division=0),
    "recall": recall_score(y_test, test_predictions, zero_division=0),
    "f1": f1_score(y_test, test_predictions, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_probabilities),
    "average_precision": average_precision_score(y_test, test_probabilities),
    "log_loss": log_loss(y_test, test_probabilities),
}]).set_index("k")
print("Final test metrics:")
print(test_results.round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
RocCurveDisplay.from_predictions(y_test, test_probabilities, ax=axes[0], name=f"k={best_k}")
axes[0].plot([0, 1], [0, 1], "k--")
PrecisionRecallDisplay.from_predictions(y_test, test_probabilities, ax=axes[1], name=f"k={best_k}")
axes[1].axhline(y_test.mean(), color="black", linestyle="--")
ConfusionMatrixDisplay.from_predictions(
    y_test, test_predictions, labels=[0, 1], display_labels=["Negative", "Positive"],
    ax=axes[2], colorbar=False, cmap="Blues",
)
axes[0].set_title("Official test ROC curve")
axes[1].set_title("Official test precision-recall curve")
axes[2].set_title("Official test confusion matrix")
plt.tight_layout()
plt.show()
